# JedAI 50M — Train on a Free Colab T4

Trains the **~50M parameter** base model on `wikitext-103` + your books/websites corpus.

**First:** `Runtime → Change runtime type → T4 GPU → Save`.

### How this notebook is organised
- **A. Setup** — run every session (clone, install, mount Google Drive).
- **B. Prepare data — FIRST RUN ONLY** — download + clean + tokenize, then cache to Drive (~10–15 min).
- **C. Restore data — RESUMING A LATER SESSION** — copy the cached data back from Drive (~30 s). Run **B or C, not both.**
- **D. Train** — checkpoints go to your Drive and **auto-resume**, so a disconnect is no big deal.
- **E. Generate / F. Chat finetune / G. Save.**

> A full 20,000-step run is ~15–18 h on a T4 — longer than one free session. That's fine: when Colab disconnects, just reconnect and re-run **A → C → D** and training picks up from the last Drive checkpoint.

## A. Setup (run every session)

In [ ]:
!git clone https://github.com/jadetindoy/jed-ai.git
%cd jed-ai
# Colab already ships a CUDA build of torch; add the light extras only.
!pip install -q tokenizers datasets rich pyyaml tqdm

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('NO GPU -> Runtime > Change runtime type > T4 GPU. (CPU works but is far too slow for 50M.)')

In [ ]:
# Mount Google Drive so checkpoints + tokenized data survive disconnects.
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/jedai')
CKPT_DIR   = DRIVE_ROOT / 'checkpoints' / '50m_t4'   # checkpoints persist here
DATA_CACHE = DRIVE_ROOT / 'tokenized'                # train.bin/val.bin cache
CKPT_DIR.mkdir(parents=True, exist_ok=True)
DATA_CACHE.mkdir(parents=True, exist_ok=True)
print('Checkpoints ->', CKPT_DIR)
print('Data cache  ->', DATA_CACHE)

## B. Prepare data — FIRST RUN ONLY

Downloads wikitext-103 and tokenizes everything, then caches the result to Drive. Skip this whole section on later sessions — use **C** instead.

In [ ]:
# 1) Download wikitext-103 (~520 MB) into data/raw/. Your committed .jsonl
#    books/websites are already in data/raw/ from the clone.
!python -m data.download_dataset --dataset wikitext-103

In [ ]:
# 2) Clean (handles .txt AND .jsonl, dedups across files). Dialogue stays out.
!python -m data.clean --input_dir data/raw --output_dir data/cleaned

In [ ]:
# 3) Train the 16k BPE tokenizer (includes <|user|>/<|assistant|> chat tokens).
!python -m tokenizer.train --input_dir data/cleaned --vocab_size 16000 --output_dir tokenizer

In [ ]:
# 4) Tokenize cleaned corpus -> data/tokenized/{train,val}.bin (~125M tokens).
!python -m data.tokenize_dataset --input_dir data/cleaned --output_dir data/tokenized --tokenizer tokenizer/tokenizer.json --val_ratio 0.1

In [ ]:
# 5) Cache tokenized data + tokenizer to Drive so future sessions skip steps 1-4.
import shutil
for f in ['train.bin', 'val.bin', 'metadata.json']:
    shutil.copy(f'data/tokenized/{f}', DATA_CACHE / f)
shutil.copy('tokenizer/tokenizer.json', DATA_CACHE / 'tokenizer.json')
print('Cached to', DATA_CACHE, '->', [p.name for p in DATA_CACHE.iterdir()])

## C. Restore data — RESUMING A LATER SESSION

Run this **instead of B** once the cache exists in Drive. Copies the tokenized data back in seconds.

In [ ]:
import shutil
from pathlib import Path
Path('data/tokenized').mkdir(parents=True, exist_ok=True)
for f in ['train.bin', 'val.bin', 'metadata.json']:
    shutil.copy(DATA_CACHE / f, f'data/tokenized/{f}')
shutil.copy(DATA_CACHE / 'tokenizer.json', 'tokenizer/tokenizer.json')
print('Restored tokenized data from Drive cache. Ready to train.')

## D. Train the 50M model

`--output_dir` points at Drive, and the trainer **auto-resumes** from the latest checkpoint there. Re-run this cell after any disconnect to continue.

OOM? Edit `configs/50m_t4.yaml` and set `batch_size: 8` (raise `grad_accum_steps` to 8 to keep the effective batch).

In [ ]:
!python training/trainer.py --config configs/50m_t4.yaml --output_dir "$(python -c 'from pathlib import Path; print(Path("/content/drive/MyDrive/jedai/checkpoints/50m_t4"))')"

## E. Generate text from the latest checkpoint

In [ ]:
from pathlib import Path
from inference.generate import Generator

ckpts = sorted(CKPT_DIR.glob('ckpt_*.pt'))
assert ckpts, 'No checkpoint in Drive yet — let training run past the first save_every (1000 steps).'
print('Using', ckpts[-1].name)
gen = Generator(model_path=str(ckpts[-1]), tokenizer_path='tokenizer/tokenizer.json')
print(gen.generate('The history of artificial intelligence', max_new_tokens=80, temperature=0.8, top_k=40))

## F. (Optional) Chat finetune

Run **after** the base model is trained. Turns the text-continuer into a chatbot using `data/dialogue/conversations.txt`. Saves to Drive too.

In [ ]:
from pathlib import Path
ckpts = sorted(CKPT_DIR.glob('ckpt_*.pt'))
base = ckpts[-1]
chat_out = '/content/drive/MyDrive/jedai/checkpoints/chat'
!python -m training.finetune_chat --base_checkpoint "{base}" --data data/dialogue/conversations.txt --tokenizer tokenizer/tokenizer.json --output_dir "{chat_out}" --epochs 3

## G. Save a checkpoint to your computer (optional)

It's already safe in Drive, but you can also pull a copy down.

In [ ]:
from google.colab import files
from pathlib import Path
ckpts = sorted(CKPT_DIR.glob('ckpt_*.pt'))
files.download(str(ckpts[-1]))